# Water Quality Prediction: Benchmark Notebook 

In this notebook, we demonstrate a basic workflow that serves as a foundation for the challenge. The model has been developed to predict **water quality parameters** using features derived from the **Landsat** and **TerraClimate** datasets. Specifically, four spectral bands — **SWIR22** (Shortwave Infrared 2), **NIR** (Near Infrared), **Green**, and **SWIR16** (Shortwave Infrared 1) — were utilized from Landsat, along with derived spectral indices such as **NDMI** (Normalized Difference Moisture Index) and **MNDWI** (Modified Normalized Difference Water Index). In addition, the **PET** (Potential Evapotranspiration) variable was incorporated from the **TerraClimate** dataset to account for climatic influences on water quality.

The dataset spans a five-year period from **2011 to 2015**. Using **API-based data extraction** methods, both Landsat and TerraClimate features were retrieved directly from the [Microsoft Planetary Computer portal](https://planetarycomputer.microsoft.com).

These combined spectral, index-based, and climatic features were used as predictors in a regression model to estimate three key water quality parameters: **Total Alkalinity (TA)**, **Electrical Conductance (EC)**, and **Dissolved Reactive Phosphorus (DRP)**.

## Environment preparation

In [ ]:
!pip install uv
!uv pip install -r ../requirements.txt 

In [1]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Data manipulation and analysis
import numpy as np
import pandas as pd
from IPython.display import display

# Multi-dimensional arrays and datasets (e.g., NetCDF, Zarr)
import xarray as xr

# Geospatial raster data handling with CRS support
import rioxarray as rxr

# Raster operations and spatial windowing
import rasterio
from rasterio.windows import Window

# Machine Learning
from xgboost import XGBRegressor
from xgboost import plot_importance
                                                     # CV that respects groups
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score
from sklearn.cluster import KMeans # Clusterization
from sklearn.metrics import r2_score, mean_squared_error
import optuna
import optuna.visualization as vis

from datetime import date
from tqdm import tqdm
import re

# Importing functions
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df, combine_datasets

In [ ]:
# Path
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/submissions/'

# Features that don't need to be converted to float, and do not go on model building
not_to_convert_data = [
    'Latitude',
    'Longitude',
    'Sample Date',
    'location_id',
    'soil_name',
    'kmeans_region'
]

# This cols cannot be lt0
cols_to_fix = [
    'Total Alkalinity',
    'Electrical Conductance',
    'Dissolved Reactive Phosphorus'
]

# Features not included on train
cols_to_drop = cols_to_fix + [
    'Sample Date',

    # High correlation
    'SI', 'blue', 'red', # Almost same as green
    'swir22', # Almost same as swir16
    'srad', # Almost same as pet
    'month', 'quarter', 'year', # Ciclic date features (sin/cos) is better
    # 'soil' # Impact much more than other features - TEST
]

# Features to drop after grouping
# To prevent model from overfitting in lat + lon, we created
# groups for eatch lat + lon combination, and guarantee that 
# the groups in the training dataset are NOT in the testing 
# dataset, ensuring that we validate in rivers not seen before.
spatial_cols = [
    'Latitude',
    'Longitude',
    'location_id'
]

In [ ]:
# Train data
train_features = (
    pd.read_csv(f'../data/processed/train_features.csv')
)
# train_features['kmeans_region'] = train_features['kmeans_region'].astype('category')
# train_features['soil_name'] = train_features['soil_name'].astype('category')

# Test data
val_features = (
    pd.read_csv(f'../data/processed/val_features.csv')
)
# val_features['kmeans_region'] = val_features['kmeans_region'].astype('category')
# val_features['soil_name'] = val_features['soil_name'].astype('category')

# Submission df
base_submission_df = (
    pd.read_csv(f'../data/raw/submission_template.csv')
)

In [ ]:
for df in [train_features, val_features]:
    for col in df.columns:
        if not col in not_to_convert_data:
            df[col] = df[col].astype(float)

## Data transformation

In [7]:
adjusted_combined_train_df = (
    train_features.fillna(train_features.median(numeric_only=True))
)

adjusted_combined_test_df = (
    val_features.fillna(val_features.median(numeric_only=True))
)

## Model Building

In [9]:
def split_data(x, y, test_size=0.3, random_state=42):
    '''
    Guarantees that the model will be tested on new rivers, and 
    not in the same as on train
    '''
    # Unique ID
    groups = x['location_id']

    # To sort whole locals and not 'each line'
    # Only cut one time (train vs test)
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=test_size,
        random_state=random_state
    )

    # Take indexes (guarantee that rivers will not repeat themselves)
    train_idx, test_idx = next(splitter.split(x, y, groups))
    
    return x.iloc[train_idx], x.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx], groups.iloc[train_idx]

def train_model(x_train, y_train, groups_train):
    '''
    XGBRegressor optimized with optuna
    handles NaN by default and dont need scaled data
    Receives groups_train to make GPS stop leaking
    '''
    # GroupKFold
    gkf = GroupKFold(n_splits=7)

    # Optuna optimization
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 1500),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 8),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
            'subsample': trial.suggest_float('subsample', 0.6, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
            'gamma': trial.suggest_float('gamma', 0, 5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 5, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10, log=True),
            # Fixed
            'tree_method': 'hist',
            'random_state': 42,
            'n_jobs': -1,
            'enable_categorical': True
        }
    
        # Execution model
        model = XGBRegressor(**params)
    
        # Score for optuna
        score = cross_val_score(
            model,
            x_train,
            y_train,
            groups=groups_train,
            cv=gkf,
            scoring='neg_root_mean_squared_error',
            n_jobs=-1
        ).mean()
    
        return score

    print('Initializing optuna tuning')
    study = optuna.create_study(
        direction='maximize', 
        sampler=optuna.samplers.TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=150)

    # Feedback
    best_params = study.best_params
    print(f"\n\nBEST PARAMETERS FOUND: {best_params}")

    # Readding fixed parameters
    best_params["tree_method"] = "hist"
    best_params["random_state"] = 42
    best_params["n_jobs"] = -1
    best_params["enable_categorical"] = True

    # Best model
    best_xgb = XGBRegressor(**best_params)
    
    print("\nTraining best model...")
    best_xgb.fit(x_train, y_train)

    return best_xgb, study
        
def evaluate_model(model, x_test, y_true, dataset_name='Test'):
    '''
    model = 'fitted' model with training data
    x_test = missing data
    y_true = real data

    returns the related feature forecasted using the x_test
    and validates it before using on the final database
    '''
    # Forecasting
    y_pred = model.predict(x_test)

    # Evaluating model
    r2 = r2_score(y_true, y_pred) # 1.0 is perfect, 0.0 is horrible
    rmse = np.sqrt(mean_squared_error(y_true, y_pred)) # Less is better

    print(f'\n{dataset_name} Evaluation:')
    print(f'R²: {r2:.3f}')
    print(f'RMSE: {rmse:.3f}')

    # Plotting results
    plt.figure(figsize=(8, 4))

    # X axis -> real
    # Y axis -> forecast
    sns.scatterplot(x=y_true, y=y_pred, alpha=0.5)

    # If a dot is above the line -> superestimated
    # If a dot is under the line -> underestimated
    max_val = max(y_true.max(), y_pred.max())
    min_val = min(y_true.min(), y_pred.min())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)

    # Plot
    plt.xlabel('Real')
    plt.ylabel('Forecasted')
    plt.title(f'Real vs Forecasted ({dataset_name}) - R²: {r2:.2f} | RMSE: {rmse:.2f}')
    plt.show()
    
    return y_pred

In [10]:
def run_pipeline(x, y, param_name='Parameter'):
    print(f'\n{'='*60}')
    print(f'Training Model for {param_name}')
    print(f'{'='*60}')
    
    # Split data - needs to be improved (group segregation)
    x_train, x_test, y_train, y_test, groups_train = split_data(x, y)

    # Removing latitude and longitude
    x_train_clean = x_train.drop(columns=spatial_cols, errors='ignore')
    x_test_clean = x_test.drop(columns=spatial_cols, errors='ignore')
    
    # Training model
    model, study = train_model(x_train_clean, y_train, groups_train)
    
    # Evaluate (in-sample)
    y_train_pred = evaluate_model(model, x_train_clean, y_train, 'Train')
    
    # Evaluate (out-sample)
    y_test_pred = evaluate_model(model, x_test_clean, y_test, 'Test')

    # Shows feature importance
    try:
        plt.figure(figsize=(10, 6))
        plot_importance(
            model,
            max_num_features=15,
            height=0.5,
            importance_type='weight',
            title=f'Features importance - {param_name}'
        )
        plt.show()

        print('\n\nOptuna Hyperparameter Importance')
        vis.plot_param_importances(study)
    except Exception as e:
        print("Could not plot importance (maybe no features left?)")
    
    return model

In [11]:
# In this step, we apply the complete modeling pipeline to each of the three selected water quality parameters:
#     Total Alkalinity, Electrical Conductance, and Dissolved Reactive Phosphorus.
    
# The input feature set (`X`) remains the same across all three models, while the target variable (`y`) changes for each parameter. 
# For every parameter, the `run_pipeline()` function is executed, which handles data preprocessing, model training, and both in-sample
# and out-of-sample evaluation. This ensures a consistent workflow and allows for a fair comparison of model performance across different
# water quality indicators.

# Extracts only desired features
x = train_features.drop(columns=cols_to_drop, errors='ignore')
submission_features = val_features.drop(columns=cols_to_drop, errors='ignore')

# Targets
y_TA = train_features['Total Alkalinity']
y_EC = train_features['Electrical Conductance']
y_DRP = train_features['Dissolved Reactive Phosphorus']

In [ ]:
# It uses the log of the feature to distribute it better
model_TA = run_pipeline(x, np.log1p(y_TA), 'Total Alkalinity')
model_EC = run_pipeline(x, np.log1p(y_EC), 'Electrical Conductance')
model_DRP = run_pipeline(x, np.log1p(y_DRP), 'Dissolved Reactive Phosphorus')

## Submission

In [20]:
# Removing latitude and longitude
submission_features_clean = submission_features.drop(columns=spatial_cols, errors='ignore')

# Guarantee correct order
cols_correct_order = model_TA.feature_names_in_
submission_features_final = submission_features_clean[cols_correct_order]

# Predicting for every feature
# Reverts the log value using exponential
pred_TA_submission = np.expm1(model_TA.predict(submission_features_final))
pred_EC_submission = np.expm1(model_EC.predict(submission_features_final))
pred_DRP_submission = np.expm1(model_DRP.predict(submission_features_final))

In [21]:
# Overwrite null values from template
final_submission = base_submission_df
final_submission['Total Alkalinity'] = pred_TA_submission
final_submission['Electrical Conductance'] = pred_EC_submission
final_submission['Dissolved Reactive Phosphorus'] = pred_DRP_submission

# If a value is lt0, replace it with 0 (cannot be negative)
final_submission[cols_to_fix] = (
    final_submission[cols_to_fix].clip(lower=0)
)

# Verifies possible Nan or infinity
n_nans = final_submission[cols_to_fix].isna().sum().sum()
if n_nans > 0:
    print(f"URGENT: Found {n_nans} null values! Replacing with 0...")
    final_submission = final_submission.fillna(0)

# Show result
print("\nSubmission statistics:")
display(final_submission[cols_to_fix].describe())

In [23]:
# Takes all trys
next_try = max([
    int(re.sub(r'\D', '', file_name))
    for file_name
    in os.listdir('../submissions')
]) + 1

# Dumping the predictions into a csv file.
save_df(final_submission, f'submission_v{next_try}.csv', save_path)